# NB03 — Deep Momentum Network

In [ ]:
from pathlib import Path
import sys
import warnings
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)

ROOT = Path.cwd()
if not (ROOT / "configs" / "default.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.nb03_dmn import (
    resolve_project_root,
    load_nb03_inputs, build_feature_matrix,
    input_summary, feature_summary, select_example_ticker,
    model_architecture_table, FEATURE_COLS, SEQ_LEN, HIDDEN_SIZE,
    MAX_EPOCHS, MAX_TRAIN_SEQ, PATIENCE,
    plot_feature_correlation, plot_fold_sharpe,
    plot_position_on_ticker, plot_position_distribution,
    run_walk_forward, walk_forward_splits,
    save_nb03_outputs,
)

ROOT = resolve_project_root(ROOT)

## 1. Inputs & Feature Matrix

In [ ]:
data         = load_nb03_inputs(ROOT)
panel        = data["panel"]
cpd_features = data["cpd_features"]

feat         = build_feature_matrix(panel, cpd_features)
feature_cols = [c for c in FEATURE_COLS if c in feat.columns]

display(input_summary(feat))

## 2. Feature Set

In [ ]:
display(feature_summary(feat))
plot_feature_correlation(feat, feature_cols).show()

## 3. Architecture

In [ ]:
display(model_architecture_table(len(feature_cols)))

## 4. Walk-Forward Training

In [ ]:
splits = walk_forward_splits(feat)
display(pd.DataFrame([
    {"fold": i + 1, "train": f"2006 → {sp['train_end'].year}", "test": sp["test_year"]}
    for i, sp in enumerate(splits)
]))

print(f"τ={SEQ_LEN}j · hidden={HIDDEN_SIZE} · max_epochs={MAX_EPOCHS} · patience={PATIENCE}")
print(f"Séquences/fold : cap {MAX_TRAIN_SEQ:,} (sous-échantillonnage reproducible, univers complet)")
print()

positions, fold_metrics = run_walk_forward(feat, feature_cols=feature_cols, verbose=True)

display(fold_metrics)
plot_fold_sharpe(fold_metrics).show()

## 5. Positions

In [ ]:
example_ticker = select_example_ticker(feat)
plot_position_on_ticker(feat, positions, example_ticker).show()
plot_position_distribution(positions).show()

## 6. Save

In [ ]:
display(save_nb03_outputs(ROOT, positions, fold_metrics))